In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

import joblib

In [2]:
data = {
    "total_students": [
        500, 450, 400, 350, 300,
        250, 200, 150, 100, 80,
        600, 520, 430, 370, 280
    ],

    "attendance_rate": [
        95, 90, 85, 80, 75,
        70, 65, 60, 55, 50,
        97, 92, 86, 78, 72
    ],

    "assignment_completion_rate": [
        98, 95, 90, 85, 80,
        75, 70, 65, 60, 55,
        99, 94, 88, 82, 76
    ],

    "active_students": [
        480, 420, 370, 320, 260,
        210, 160, 120, 80, 50,
        590, 490, 380, 300, 220
    ],

    "completion_rate": [
        96, 92, 88, 82, 78,
        72, 65, 60, 55, 45,
        98, 94, 87, 80, 70
    ]
}


df = pd.DataFrame(data)

df.head()

,total_students,attendance_rate,assignment_completion_rate,active_students,completion_rate
0,500,95,98,480,96
1,450,90,95,420,92
2,400,85,90,370,88
3,350,80,85,320,82
4,300,75,80,260,78


In [3]:
X = df[
    [
        "total_students",
        "attendance_rate",
        "assignment_completion_rate",
        "active_students"
    ]
]

y = df["completion_rate"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


print("Training data:", X_train.shape)

print("Testing data:", X_test.shape)

Training data: (12, 4)
Testing data: (3, 4)


In [4]:
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(
    X_train,
    y_train
)


print("Completion Rate AI Model trained successfully")

Completion Rate AI Model trained successfully


In [5]:
y_pred = model.predict(
    X_test
)

mae = mean_absolute_error(
    y_test,
    y_pred
)



r2 = r2_score(
    y_test,
    y_pred
)


print(
    "Mean Absolute Error:",
    round(mae, 2)
)


print(
    "R2 Score:",
    round(r2, 2)
)

Mean Absolute Error: 5.31
R2 Score: 0.91


In [6]:
joblib.dump(
    model,
    "completion_rate_model.pkl"
)


print("Completion Rate Model saved successfully")

Completion Rate Model saved successfully


In [7]:
from fastapi import FastAPI
import joblib


app = FastAPI(
    title="Completion Rate Prediction AI API",
    description="Predict university completion rate using Machine Learning",
    version="1.0"
)


model = joblib.load(
    "completion_rate_model.pkl"
)


print("Completion Rate AI Model loaded successfully ")

Completion Rate AI Model loaded successfully 


In [8]:
@app.get("/")
def home():

    return {
        "message": "Hello World"
    }



@app.post("/completion-rate")
def completion_rate(data: dict):

    input_data = [[
        data.get("total_students"),
        data.get("attendance_rate"),
        data.get("assignment_completion_rate"),
        data.get("active_students")
    ]]


    prediction = model.predict(
        input_data
    )[0]


    if prediction >= 80:
        confidence = "High"

    elif prediction >= 50:
        confidence = "Medium"

    else:
        confidence = "Low"


    return {

        "universityId":
            data.get("universityId"),

        "predicted_completion_rate":
            round(float(prediction), 2),

        "confidence":
            confidence,

        "expected_completion_rate_this_semester":
            f"{round(float(prediction), 2)}%"
    }

In [ ]:
import nest_asyncio
import uvicorn


nest_asyncio.apply()


config = uvicorn.Config(
    app,
    host="127.0.0.1",
    port=8000,
    log_level="info"
)


server = uvicorn.Server(config)


await server.serve()

INFO:     Started server process [6960]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:59031 - "GET / HTTP/1.1" 200 OK
